# Super-clusters de sites de liaison actine

Regroupe les clusters de sites de liaison S1 et S2 en familles selon un critère de **contenance à 100%** :
deux clusters appartiennent à la même famille si l'empreinte de résidus canoniques actine de l'un est
entièrement contenue dans celle de l'autre.

Les 4 familles principales sont nommées d'après les clusters dominants : **6685_1, 6685_2, 6685_3, 6685_4**.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

## 1. Chargement des données

In [2]:
ROOT = Path("..")  # depuis notebooks/

df_all  = pd.read_csv(ROOT / "data/filtered/filtered_all_data.csv", low_memory=False)
df_int  = pd.read_csv(ROOT / "data/filtered/details/1.interactions.csv")
df_res  = pd.read_csv(ROOT / "data/filtered/details/3.interface_residues.csv")

# Garder uniquement les résidus avec position canonique connue
df_res["canon"] = pd.to_numeric(df_res["residue_number_canon_mafft"], errors="coerce")
df_res = df_res[df_res["canon"].notna()].copy()

print(f"filtered_all_data : {len(df_all)} lignes, {df_all.shape[1]} colonnes")
print(f"interactions      : {len(df_int)} lignes")
print(f"interface_residues: {len(df_res)} lignes")

filtered_all_data : 2074 lignes, 50 colonnes
interactions      : 2074 lignes
interface_residues: 96885 lignes


## 2. Mapping interaction_id → clusters S1 / S2

In [3]:
# Joindre les clusters de sites de liaison via chain_A_id / chain_B_id
# df_all a subunit_1, subunit_2 ; df_int a chain_A_id, chain_B_id, interaction_id
imap = df_int[["interaction_id", "chain_A_id", "chain_B_id"]].merge(
    df_all[[
        "subunit_1", "subunit_2",
        "s1_binding_site_cluster_data_70",
        "s2_binding_site_cluster_data_70",
        "s1_actine", "s2_actine"
    ]],
    left_on=["chain_A_id", "chain_B_id"],
    right_on=["subunit_1", "subunit_2"],
    how="left"
).drop(columns=["subunit_1", "subunit_2"])

imap["s1_actine"] = imap["s1_actine"].fillna(False).astype(bool)
imap["s2_actine"] = imap["s2_actine"].fillna(False).astype(bool)

print(imap.head(3))

   interaction_id chain_A_id chain_B_id s1_binding_site_cluster_data_70  \
0            1000     6d8c_K     6d8c_H                          6685_2   
1            1001     6d8c_K     6d8c_B                         6685_16   
2            1002     6d8c_K     6d8c_C                         6685_15   

  s2_binding_site_cluster_data_70  s1_actine  s2_actine  
0                          6685_1       True       True  
1                         15620_0       True      False  
2                         15620_1       True      False  


## 3. Calcul des empreintes de résidus canoniques actine

In [4]:
def compute_footprints(imap_subset, cluster_col, actin_chain_col):
    """
    Pour chaque cluster de sites de liaison, calcule l'ensemble des résidus
    canoniques actine contactés (union sur toutes les interactions du cluster).
    
    actin_chain_col : 'chain_A_id' si actin est S1, 'chain_B_id' si actin est S2
    """
    footprints = {}
    for cluster, grp in imap_subset.groupby(cluster_col):
        if pd.isna(cluster):
            continue
        canon_positions = set()
        for _, row in grp.iterrows():
            iid        = row["interaction_id"]
            actin_ch   = row[actin_chain_col]
            res_subset = df_res[
                (df_res["interaction_id"] == iid) &
                (df_res["chain"] == actin_ch)
            ]
            canon_positions.update(res_subset["canon"].dropna().astype(int).tolist())
        if canon_positions:
            footprints[str(cluster)] = canon_positions
    return footprints

# S1 : actin est chain_A (s1_actine=True)
imap_s1 = imap[imap["s1_actine"] & imap["s1_binding_site_cluster_data_70"].notna()]
fp_s1   = compute_footprints(imap_s1, "s1_binding_site_cluster_data_70", "chain_A_id")

# S2 : actin est chain_B (s2_actine=True)
imap_s2 = imap[imap["s2_actine"] & imap["s2_binding_site_cluster_data_70"].notna()]
fp_s2   = compute_footprints(imap_s2, "s2_binding_site_cluster_data_70", "chain_B_id")

print(f"Empreintes S1 calculées : {len(fp_s1)} clusters")
print(f"Empreintes S2 calculées : {len(fp_s2)} clusters")

Empreintes S1 calculées : 156 clusters
Empreintes S2 calculées : 16 clusters


## 4. Construction des super-clusters par contenance à 100%

In [5]:
def build_superclusters(footprints, main_clusters):
    """
    Regroupe les clusters par contenance à 100% (composantes connexes).
    Nomme chaque composante d'après le cluster principal qu'elle contient
    (dans main_clusters, par ordre de priorité). Si aucun cluster principal,
    utilise le cluster ayant la plus grande empreinte comme nom.
    
    Retourne : dict {cluster_id -> supercluster_name}
    """
    clusters = list(footprints.keys())
    n = len(clusters)
    
    # Union-Find
    parent = {c: c for c in clusters}
    
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    
    def union(x, y):
        parent[find(x)] = find(y)
    
    # Créer les arêtes de contenance
    edges = 0
    for i, a in enumerate(clusters):
        fa = footprints[a]
        for b in clusters[i+1:]:
            fb = footprints[b]
            inter = len(fa & fb)
            if inter == 0:
                continue
            # 100% containment dans un sens ou l'autre
            if inter == len(fa) or inter == len(fb):
                union(a, b)
                edges += 1
    
    print(f"  {edges} arêtes de contenance")
    
    # Regrouper par composante
    components = defaultdict(list)
    for c in clusters:
        components[find(c)].append(c)
    
    # Nommer les composantes
    cluster_to_super = {}
    for root, members in components.items():
        # Chercher un cluster principal dans les membres
        name = None
        for mc in main_clusters:  # priorité : 6685_1 > 6685_2 > 6685_3 > 6685_4
            if mc in members:
                name = mc
                break
        if name is None:
            # Pas de cluster principal : prendre celui avec la plus grande empreinte
            name = max(members, key=lambda c: len(footprints[c]))
        for m in members:
            cluster_to_super[m] = name
    
    return cluster_to_super, components


MAIN = ["6685_1", "6685_2", "6685_3", "6685_4"]

print("=== S1 ===")
c2s_s1, comp_s1 = build_superclusters(fp_s1, MAIN)
print(f"  {len(set(c2s_s1.values()))} super-clusters S1")

print("=== S2 ===")
c2s_s2, comp_s2 = build_superclusters(fp_s2, MAIN)
print(f"  {len(set(c2s_s2.values()))} super-clusters S2")

=== S1 ===
  241 arêtes de contenance
  41 super-clusters S1
=== S2 ===
  8 arêtes de contenance
  9 super-clusters S2


## 5. Vue détaillée des super-clusters

In [6]:
def show_superclusters(c2s, footprints, label):
    from collections import defaultdict
    groups = defaultdict(list)
    for c, s in c2s.items():
        groups[s].append(c)
    
    rows = []
    for super_name, members in sorted(groups.items(), key=lambda x: -len(x[1])):
        for m in sorted(members):
            rows.append({
                "type": label,
                "binding_site_cluster": m,
                "supercluster": super_name,
                "n_residues": len(footprints.get(m, set())),
                "is_main_cluster": m in ["6685_1","6685_2","6685_3","6685_4"]
            })
    return pd.DataFrame(rows)

df_sc_s1 = show_superclusters(c2s_s1, fp_s1, "S1")
df_sc_s2 = show_superclusters(c2s_s2, fp_s2, "S2")

print("=== Super-clusters S1 ===")
display(df_sc_s1.groupby("supercluster")[["binding_site_cluster","n_residues"]].agg(list))

print("\n=== Super-clusters S2 ===")
display(df_sc_s2.groupby("supercluster")[["binding_site_cluster","n_residues"]].agg(list))

=== Super-clusters S1 ===


,binding_site_cluster,n_residues
supercluster,,
6685_1,"[6685_1, 6685_208]","[37, 7]"
6685_109,[6685_109],[27]
6685_113,[6685_113],[10]
6685_121,[6685_121],[25]
6685_138,[6685_138],[28]
6685_144,[6685_144],[26]
6685_150,[6685_150],[47]
6685_151,[6685_151],[38]
6685_154,[6685_154],[13]



=== Super-clusters S2 ===


,binding_site_cluster,n_residues
supercluster,,
6685_1,"[6685_1, 6685_109, 6685_116, 6685_171, 6685_27...","[39, 31, 18, 14, 27, 13]"
6685_19,[6685_19],[18]
6685_2,[6685_2],[30]
6685_23,[6685_23],[22]
6685_3,"[6685_3, 6685_64]","[36, 7]"
6685_4,"[6685_17, 6685_4]","[13, 50]"
6685_57,[6685_57],[9]
6685_59,[6685_59],[10]
6685_61,[6685_61],[11]


## 6. Ajout des colonnes dans filtered_all_data.csv

In [7]:
df_out = df_all.copy()

# s1_supercluster : rempli uniquement si s1_actine=True
def map_s1(row):
    if not row.get("s1_actine", False):
        return None
    c = str(row["s1_binding_site_cluster_data_70"]) if pd.notna(row["s1_binding_site_cluster_data_70"]) else None
    return c2s_s1.get(c) if c else None

def map_s2(row):
    if not row.get("s2_actine", False):
        return None
    c = str(row["s2_binding_site_cluster_data_70"]) if pd.notna(row["s2_binding_site_cluster_data_70"]) else None
    return c2s_s2.get(c) if c else None

df_out["s1_supercluster"] = df_out.apply(map_s1, axis=1)
df_out["s2_supercluster"] = df_out.apply(map_s2, axis=1)

print("s1_supercluster — distribution :")
print(df_out["s1_supercluster"].value_counts(dropna=False).head(10))
print("\ns2_supercluster — distribution :")
print(df_out["s2_supercluster"].value_counts(dropna=False).head(10))

s1_supercluster — distribution :
s1_supercluster
6685_3      729
6685_2      490
6685_4      402
6685_1      218
6685_21      62
6685_258     24
6685_121     23
6685_263     18
6685_178     10
6685_144      9
Name: count, dtype: int64

s2_supercluster — distribution :
s2_supercluster
None       800
6685_1     490
6685_3     341
6685_2     217
6685_4     165
6685_23     54
6685_19      4
6685_57      1
6685_59      1
6685_61      1
Name: count, dtype: int64


In [8]:
# Sauvegarder filtered_all_data.csv avec les nouvelles colonnes
out_path = ROOT / "data/filtered/filtered_all_data.csv"
df_out.to_csv(out_path, index=False)
print(f"Sauvegardé : {out_path}")
print(f"Nouvelles colonnes : s1_supercluster, s2_supercluster")

Sauvegardé : ../data/filtered/filtered_all_data.csv
Nouvelles colonnes : s1_supercluster, s2_supercluster


## 7. CSV de traçabilité

In [9]:
df_trace = pd.concat([df_sc_s1, df_sc_s2], ignore_index=True)
df_trace = df_trace.sort_values(["type", "supercluster", "n_residues"], ascending=[True, True, False])

trace_path = ROOT / "data/filtered/binding_site_superclusters.csv"
df_trace.to_csv(trace_path, index=False)
print(f"CSV de traçabilité : {trace_path}")
display(df_trace)

CSV de traçabilité : ../data/filtered/binding_site_superclusters.csv


,type,binding_site_cluster,supercluster,n_residues,is_main_cluster
115,S1,6685_1,6685_1,37,True
116,S1,6685_208,6685_1,7,False
123,S1,6685_109,6685_109,27,False
124,S1,6685_113,6685_113,10,False
125,S1,6685_121,6685_121,25,False
...,...,...,...,...,...
163,S2,6685_4,6685_4,50,True
162,S2,6685_17,6685_4,13,False
169,S2,6685_57,6685_57,9,False
170,S2,6685_59,6685_59,10,False


## 8. Résumé des empreintes des 4 familles principales

In [10]:
print("Empreintes des 4 familles principales (résidus canoniques actine) :")
for m in MAIN:
    fp_s1_m = fp_s1.get(m, set())
    fp_s2_m = fp_s2.get(m, set())
    print(f"\n{m}")
    if fp_s1_m:
        print(f"  S1 : {len(fp_s1_m)} résidus — min={min(fp_s1_m)}, max={max(fp_s1_m)}")
        print(f"       {sorted(fp_s1_m)[:10]}{'...' if len(fp_s1_m)>10 else ''}")
    if fp_s2_m:
        print(f"  S2 : {len(fp_s2_m)} résidus — min={min(fp_s2_m)}, max={max(fp_s2_m)}")
        print(f"       {sorted(fp_s2_m)[:10]}{'...' if len(fp_s2_m)>10 else ''}")

Empreintes des 4 familles principales (résidus canoniques actine) :

6685_1
  S1 : 37 résidus — min=43, max=275
       [43, 44, 45, 65, 66, 67, 68, 69, 70, 71]...
  S2 : 39 résidus — min=43, max=275
       [43, 44, 45, 65, 66, 67, 68, 69, 70, 71]...

6685_2
  S1 : 39 résidus — min=76, max=380
       [76, 77, 79, 80, 81, 114, 115, 116, 117, 118]...
  S2 : 30 résidus — min=77, max=380
       [77, 79, 81, 114, 115, 116, 117, 118, 120, 176]...

6685_3
  S1 : 43 résidus — min=42, max=254
       [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]...
  S2 : 36 résidus — min=42, max=252
       [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]...

6685_4
  S1 : 52 résidus — min=120, max=380
       [120, 137, 139, 140, 143, 144, 146, 147, 150, 151]...
  S2 : 50 résidus — min=108, max=380
       [108, 137, 139, 140, 143, 144, 146, 147, 150, 151]...
